<a href="https://colab.research.google.com/github/Bborub/Music/blob/main/GarmanKlassVolatility_YangZhangVolatility_072625.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
! pip install yfinance
! pip install volatility3
from volatility3 import volest
import yfinance as yf

# data
symbol = 'JPM'
bench = 'SPY'
estimator = 'GarmanKlass'

# estimator windows
window = 30
windows = [30, 60, 90, 120]
quantiles = [0.25, 0.75]
bins = 100
normed = True

# use the yahoo helper to correctly format data from finance.yahoo.com
jpm_price_data = yf.Ticker(symbol).history(period="5y")
jpm_price_data.symbol = symbol
spx_price_data = yf.Ticker(bench).history(period="5y")
spx_price_data.symbol = bench

# initialize class
vol = volest.VolatilityEstimator(
    price_data=jpm_price_data,
    estimator=estimator,
    bench_data=spx_price_data
)

# call plt.show() on any of the below...
_, plt = vol.cones(windows=windows, quantiles=quantiles)
_, plt = vol.rolling_quantiles(window=window, quantiles=quantiles)
_, plt = vol.rolling_extremes(window=window)
_, plt = vol.rolling_descriptives(window=window)
_, plt = vol.histogram(window=window, bins=bins, normed=normed)

_, plt = vol.benchmark_compare(window=window)
_, plt = vol.benchmark_correlation(window=window)

# ... or create a pdf term sheet with all metrics in term-sheets/
vol.term_sheet(
    window,
    windows,
    quantiles,
    bins,
    normed
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 6.3 MB/s eta 0:00:00


ImportError: cannot import name 'volest' from 'volatility3' (/usr/local/lib/python3.11/dist-packages/volatility3/__init__.py)

In [7]:
import datetime
import os

import pandas
import numpy
from scipy.stats import norm
import statsmodels.api as sm
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from volatility3 import models

ESTIMATORS = [
    'GarmanKlass',
    'HodgesTompkins',
    'Kurtosis',
    'Parkinson',
    'Raw',
    'RogersSatchell',
    'Skew',
    'YangZhang'
]
PRICE_COLUMNS = {
    'Open',
    'High',
    'Low',
    'Close'
}


def array_to_dataframe(ndarray):
    return pandas.DataFrame(
        ndarray,
        columns=['Open', 'High', 'Low', 'Close']
    )


class VolatilityEstimator(object):

    def __init__(self, price_data, estimator, bench_data=None):
        """Constructor for volatility estimators

        Parameters
        ----------
        price_data: pandas.DataFrame or numpy.ndarray
            If pandas.DataFrame, must include columns Open, High, Low, Close. Also
            must include property symbol with the symbol we're working with. If
            numpy.ndarray, must be of shape (r, 4) with columns in order of open,
            high, low, close prices. If numpy.ndarray, will be coerced to pandas.DataFrame
            with no date data
        estimator : string
            Estimator estimator; valid arguments are:
                "GarmanKlass", "HodgesTompkins", "Kurtosis", "Parkinson", "Raw",
                "RogersSatchell", "Skew", "YangZhang"
        """

        if not isinstance(price_data, numpy.ndarray) and not \
                isinstance(price_data, pandas.DataFrame):
            raise ValueError('price_data must be of type numpy.ndarray or pandas.DataFrame')
        if isinstance(price_data, numpy.ndarray) and price_data.shape[0] != 4:
            raise ValueError('price_data of type numpy.ndarray shape of (r, 4)')
        if isinstance(price_data, pandas.DataFrame) and not \
                PRICE_COLUMNS.issubset(price_data.columns):
            raise ValueError('price_data requires Open, High, Low, Close')
        if price_data.symbol is None or price_data.symbol == '':
            raise ValueError('Symbol required as property of price_data')
        if estimator not in ESTIMATORS:
            raise ValueError('Acceptable volatility model is required')

        if isinstance(price_data, numpy.ndarray):
            price_data = array_to_dataframe(price_data)
            price_data.symbol = '-NA-'
            start = price_data.index[0]
            end = price_data.index[0]
        else:
            start = price_data.index[0].to_pydatetime().strftime('%Y-%m-%d')
            end = price_data.index[-1].to_pydatetime().strftime('%Y-%m-%d')

        if bench_data is not None:
            if price_data.shape != bench_data.shape:
                raise ValueError('price_data and bench_data must be same shape')
            if not isinstance(bench_data, numpy.ndarray) and not \
                    isinstance(bench_data, pandas.DataFrame):
                raise ValueError('bench_data must be of type numpy.ndarray or pandas.DataFrame')
            if isinstance(bench_data, numpy.ndarray) and bench_data.shape[0] != 4:
                raise ValueError('bench_data of type numpy.ndarray shape of (r, 4)')
            if isinstance(bench_data, pandas.DataFrame) and not \
                    PRICE_COLUMNS.issubset(bench_data.columns):
                raise ValueError('bench_data requires Open, High, Low, Close')
            if bench_data.symbol is None or bench_data.symbol == '':
                raise ValueError('Symbol required as property of bench_data')

            # bench_data = bench_data.loc[start:end]

            if isinstance(bench_data, numpy.ndarray):
                bench_data = array_to_dataframe(bench_data)
                bench_data.symbol = '-NA-'

            self._bench_data = bench_data
            self._bench_symbol = bench_data.symbol

        self._price_data = price_data
        self._symbol = price_data.symbol
        self._start = start
        self._end = end
        self._estimator = estimator

        matplotlib.rc('image', origin='upper')

        matplotlib.rcParams['font.size'] = '11'

        matplotlib.rcParams['grid.color'] = 'lightgrey'
        matplotlib.rcParams['grid.linestyle'] = '-'

        matplotlib.rcParams['figure.subplot.left'] = 0.1
        matplotlib.rcParams['figure.subplot.bottom'] = 0.13
        matplotlib.rcParams['figure.subplot.right'] = 0.9
        matplotlib.rcParams['figure.subplot.top'] = 0.9

    def _get_estimator(self, window, price_data, clean=True):
        """Selector for volatility estimator

        Parameters
        ----------
        window : int
            Rolling window for which to calculate the estimator
        clean : boolean
            Set to True to remove the NaNs at the beginning of the series

        Returns
        -------
        y : pandas.DataFrame
            Estimator series values
        """

        return getattr(models, self._estimator).get_estimator(
            price_data=price_data,
            window=window,
            clean=clean
        )

    def cones(self, windows=[30, 60, 90, 120], quantiles=[0.25, 0.75]):
        """Plots volatility cones

        Parameters
        ----------
        windows : [int, int, ...]
            List of rolling windows for which to calculate the estimator cones
        quantiles : [lower, upper]
            List of lower and upper quantiles for which to plot the cones
        """

        price_data = self._price_data

        if len(windows) < 2:
            raise ValueError(
                'Two or more window periods required')
        if len(quantiles) != 2:
            raise ValueError(
                'A two element list of quantiles is required, lower and upper')
        if quantiles[0] + quantiles[1] != 1.0:
            raise ValueError(
                'The sum of the quantiles must equal 1.0')
        if quantiles[0] > quantiles[1]:
            raise ValueError(
                'The lower quantiles (first element) must be less than the upper quantile (second element)')

        max_ = []
        min_ = []
        top_q = []
        median = []
        bottom_q = []
        realized = []
        data = []

        for window in windows:

            estimator = self._get_estimator(
                window=window,
                price_data=price_data
            )

            max_.append(estimator.max())
            top_q.append(estimator.quantile(quantiles[1]))
            median.append(estimator.median())
            bottom_q.append(estimator.quantile(quantiles[0]))
            min_.append(estimator.min())
            realized.append(estimator[-1])

            data.append(estimator)

        if self._estimator is "Skew" or self._estimator is "Kurtosis":
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        # figure
        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        bottom, height = 0.2, 0.7
        left_h = left+width+0.02
        rect_cones = [left, bottom, width, height]
        rect_box = [left_h, bottom, 0.17, height]
        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        # set the plots
        cones.plot(windows, max_, label="Max")
        cones.plot(windows, top_q, label=str(int(quantiles[1]*100)) + " Prctl")
        cones.plot(windows, median, label="Median")
        cones.plot(windows, bottom_q, label=str(int(quantiles[0]*100)) + " Prctl")
        cones.plot(windows, min_, label="Min")
        cones.plot(windows, realized, 'r-.', label="Realized")

        # set the x ticks and limits
        cones.set_xticks(windows)
        cones.set_xlim((windows[0]-5, windows[-1]+5))

        # set and format the y-axis labels
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))

        # turn on the grid
        cones.grid(True, axis='y', which='major', alpha=0.5)

        # set the title
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')

        # set the legend
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)

        # box plot
        box.boxplot(data, notch=1, sym='+')
        box.plot([i for i in range(1, len(windows)+1)], realized, color='r', marker='*', markeredgecolor='k')

        # set and format the y-axis labels
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))

        # move the y-axis ticks on the right side
        box.yaxis.tick_right()

        # turn on the grid
        box.grid(True, axis='y', which='major', alpha=0.5)

        return fig, plt

    def rolling_quantiles(self, window=30, quantiles=[0.25, 0.75]):
        """Plots rolling quantiles of volatility

        Parameters
        ----------
        window : int
            Rolling window for which to calculate the estimator
        quantiles : [lower, upper]
            List of lower and upper quantiles for which to plot
        """

        price_data = self._price_data

        if len(quantiles) != 2:
            raise ValueError(
                'A two element list of quantiles is required, lower and upper')
        if quantiles[0] + quantiles[1] != 1.0:
            raise ValueError(
                'The sum of the quantiles must equal 1.0')
        if quantiles[0] > quantiles[1]:
            raise ValueError(
                'The lower quantiles (first element) must be less than the upper quantile (second element)')

        estimator = self._get_estimator(
            window=window,
            price_data=price_data
        )
        date = estimator.index

        top_q = estimator.rolling(window=window, center=False).quantile(quantiles[1])
        median = estimator.rolling(window=window, center=False).median()
        bottom_q = estimator.rolling(window=window, center=False).quantile(quantiles[0])
        realized = estimator
        last = estimator[-1]

        if self._estimator is "Skew" or self._estimator is "Kurtosis":
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        # figure
        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        bottom, height = 0.2, 0.7
        left_h = left+width+0.02

        rect_cones = [left, bottom, width, height]
        rect_box = [left_h, bottom, 0.17, height]

        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        # set the plots
        cones.plot(date, top_q, label=str(int(quantiles[1]*100)) + " Prctl")
        cones.plot(date, median, label="Median")
        cones.plot(date, bottom_q, label=str(int(quantiles[0]*100)) + " Prctl")
        cones.plot(date, realized, 'r-.', label="Realized")

        # set and format the y-axis labels
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))

        # turn on the grid
        cones.grid(True, axis='y', which='major', alpha=0.5)

        # set the title
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')

        # set the legend
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)

        # box plots
        box.boxplot(realized, notch=1, sym='+')
        box.plot(1, last, color='r', marker='*', markeredgecolor='k')

        # set and format the y-axis labels
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))

        # move the y-axis ticks on the right side
        box.yaxis.tick_right()

        # turn on the grid
        box.grid(True, axis='y', which='major', alpha=0.5)

        return fig, plt

    def rolling_extremes(self, window=30):
        """Plots rolling max and min of volatility estimator

        Parameters
        ----------
        window : int
            Rolling window for which to calculate the estimator
        """

        price_data = self._price_data

        estimator = self._get_estimator(
            window=window,
            price_data=price_data
        )
        date = estimator.index
        max_ = estimator.rolling(window=window, center=False).max()
        min_ = estimator.rolling(window=window, center=False).min()
        realized = estimator
        last = estimator[-1]

        if self._estimator is "Skew" or self._estimator is "Kurtosis":
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        # figure
        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        bottom, height = 0.2, 0.7
        left_h = left+width+0.02

        rect_cones = [left, bottom, width, height]
        rect_box = [left_h, bottom, 0.17, height]

        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        # set the plots
        cones.plot(date, max_, label="Max")
        cones.plot(date, min_, label="Min")
        cones.plot(date, realized, 'r-.', label="Realized")

        # set and format the y-axis labels
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))

        # turn on the grid
        cones.grid(True, axis='y', which='major', alpha=0.5)

        # set the title
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')

        # set the legend
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)

        # box plot
        box.boxplot(realized, notch=1, sym='+')
        box.plot(1, last, color='r', marker='*', markeredgecolor='k')

        # set and format the y-axis labels
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))

        # move the y-axis ticks on the right side
        box.yaxis.tick_right()

        # turn on the grid
        box.grid(True, axis='y', which='major', alpha=0.5)

        return fig, plt

    def rolling_descriptives(self, window=30):
        """Plots rolling first and second moment of volatility estimator

        Parameters
        ----------
        window : int
            Rolling window for which to calculate the estimator
        """

        price_data = self._price_data

        estimator = self._get_estimator(
            window=window,
            price_data=price_data
        )
        date = estimator.index
        mean = estimator.rolling(window=window, center=False).mean()
        std = estimator.rolling(window=window, center=False).std()
        z_score = (estimator - mean) / std

        realized = estimator
        last = estimator[-1]

        if self._estimator is "Skew" or self._estimator is "Kurtosis":
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        # figure
        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        left_h = left+width+0.02

        rect_cones = [left, 0.35, width, 0.55]
        rect_box = [left_h, 0.15, 0.17, 0.75]
        rect_z = [left, 0.15, width, 0.15]

        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)
        z = plt.axes(rect_z)

        if self._estimator is "Skew" or self._estimator is "Kurtosis":
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        # set the plots
        cones.plot(date, mean, label="Mean")
        cones.plot(date, std, label="Std. Dev.")
        cones.plot(date, realized, 'r-.', label="Realized")

        # set and format the y-axis labels
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))

        # turn on the grid
        cones.grid(True, axis='y', which='major', alpha=0.5)

        # set the title
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')

        # shrink the plot up a bit and set the legend
        pos = cones.get_position()
        cones.set_position([pos.x0, pos.y0 + pos.height * 0.1, pos.width, pos.height * 0.9]) #
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)

        # box plot
        box.boxplot(realized, notch=1, sym='+')
        box.plot(1, last, color='r', marker='*', markeredgecolor='k')

        # set and format the y-axis labels
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))

        # move the y-axis ticks on the right side
        box.yaxis.tick_right()

        # turn on the grid
        box.grid(True, axis='y', which='major', alpha=0.5)

        # z-score set the plots
        z.plot(date, z_score, 'm-', label="Z-Score")

        # turn on the grid
        z.grid(True, axis='y', which='major', alpha=0.5)

        # create a horizontal line at y=0
        z.axhline(0, 0, 1, linestyle='-', linewidth=1.0, color='black')

        # set the legend
        z.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)

        return fig, plt

    def histogram(self, window=90, bins=100, normed=True):
        """

        Parameters
        ----------
        window : int
            Rolling window for which to calculate the estimator
        bins : int

        """

        price_data = self._price_data

        estimator = self._get_estimator(
            window=window,
            price_data=price_data
        )
        mean = estimator.mean()
        std = estimator.std()
        last = estimator[-1]

        fig = plt.figure(figsize=(8, 6))

        n, bins, patches = plt.hist(estimator, bins, normed=normed, facecolor='blue', alpha=0.25)

        if normed:
            y = norm.pdf(bins, mean, std)
            plt.plot(bins, y, 'g--', linewidth=1)

        plt.axvline(last, 0, 1, linestyle='-', linewidth=1.5, color='r')

        plt.grid(True, axis='y', which='major', alpha=0.5)
        plt.title('Distribution of ' + self._estimator +
                  ' estimator values (' + self._symbol +
                  ', daily ' + self._start + ' to ' + self._end + ')')

        return fig, plt

    def benchmark_compare(self, window=90):
        """

        Parameters
        ----------
        window : int
            Rolling window for which to calculate the estimator
        bins : int

        """

        price_data = self._price_data
        bench_data = self._bench_data

        y = self._get_estimator(
            window=window,
            price_data=price_data
        )
        x = self._get_estimator(
            window=window,
            price_data=bench_data
        )
        date = y.index

        ratio = y / x

        if self._estimator is "Skew" or self._estimator is "Kurtosis":
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        # figure
        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, .9

        rect_cones = [left, 0.4, width, .5]
        rect_box = [left, 0.15, width, 0.15]

        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        # set the plots
        cones.plot(date, y, label=self._symbol.upper())
        cones.plot(date, x, label=self._bench_symbol)

        # set and format the y-axis labels
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))

        # turn on the grid
        cones.grid(True, axis='y', which='major', alpha=0.5)

        # set the title
        cones.set_title(self._estimator + ' (' + self._symbol +
                        ' v. ' + self._bench_symbol + ', daily ' +
                        self._start + ' to ' + self._end + ')')

        # shrink the plot up a bit and set the legend
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)

        # set the plot
        box.plot(date, ratio, label=self._symbol.upper() + '/' + self._bench_symbol)

        # set the y-limits
        box.set_ylim((ratio.min() - 0.05, ratio.max() + 0.05))

        # fill the area
        box.fill_between(date, ratio, 0, color='blue', alpha=0.25)

        # set the legend
        box.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)

        return fig, plt

    def benchmark_correlation(self, window=90):
        """

        Parameters
        ----------
        window : int
            Rolling window for which to calculate the estimator
        bins : int

        """

        price_data = self._price_data
        bench_data = self._bench_data

        y = self._get_estimator(
            window=window,
            price_data=price_data
        )
        x = self._get_estimator(
            window=window,
            price_data=bench_data
        )
        date = y.index

        corr = x.rolling(window=window).corr(other=y)

        if self._estimator is "Skew" or self._estimator is "Kurtosis":
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        # figure
        fig = plt.figure(figsize=(8, 6))
        cones = plt.axes()

        # set the plots
        cones.plot(date, corr)

        # set the y-limits
        cones.set_ylim((corr.min() - 0.05, corr.max() + 0.05))

        # set and format the y-axis labels
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))

        # turn on the grid
        cones.grid(True, axis='y', which='major', alpha=0.5)

        # set the title
        cones.set_title(self._estimator + ' (Correlation of ' +
                        self._symbol + ' v. ' + self._bench_symbol +
                        ', daily ' + self._start + ' to ' + self._end + ')')

        return fig, plt

    def benchmark_regression(self, window=90):
        """

        Parameters
        ----------
        window : int
            Rolling window for which to calculate the estimator
        bins : int

        """
        price_data = self._price_data
        bench_data = self._bench_data

        y = self._get_estimator(
            window=window,
            price_data=price_data
        )
        X = self._get_estimator(
            window=window,
            price_data=bench_data
        )

        model = sm.OLS(y, X)
        results = model.fit()

        return results.summary()

    def term_sheet(
            self,
            window=30,
            windows=[30, 60, 90, 120],
            quantiles=[0.25, 0.75],
            bins=100,
            normed=True,
            open=False):

        cones_fig, cones_plt = self.cones(windows=windows, quantiles=quantiles)
        rolling_quantiles_fig, rolling_quantiles_plt = self.rolling_quantiles(window=window, quantiles=quantiles)
        rolling_extremes_fig, rolling_extremes_plt = self.rolling_extremes(window=window)
        rolling_descriptives_fig, rolling_descriptives_plt = self.rolling_descriptives(window=window)
        histogram_fig, histogram_plt = self.histogram(window=window, bins=bins, normed=normed)
        benchmark_compare_fig, benchmark_compare_plt = self.benchmark_compare(window=window)
        benchmark_corr_fig, benchmark_corr_plt = self.benchmark_correlation(window=window)
        benchmark_regression = self.benchmark_regression(window=window)

        filename = self._symbol.upper() + '_termsheet_' + datetime.datetime.today().strftime("%Y%m%d") + '.pdf'
        fn = os.path.abspath(os.path.join(u'..', u'term-sheets', filename))
        pp = PdfPages(fn)

        pp.savefig(cones_fig)
        pp.savefig(rolling_quantiles_fig)
        pp.savefig(rolling_extremes_fig)
        pp.savefig(rolling_descriptives_fig)
        pp.savefig(histogram_fig)
        pp.savefig(benchmark_compare_fig)
        pp.savefig(benchmark_corr_fig)

        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111)
        ax.text(
            0, .2,
            benchmark_regression,
            family='monospace',
            fontsize=9
        )

        plt.axis('off')
        fig.tight_layout()
        pp.savefig(fig)
        pp.close()

        print('%s output complete' % filename)

<>:193: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:193: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:285: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:285: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:359: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:359: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:434: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:434: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:453: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:453: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:570: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:570: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:645: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:645: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:193: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:193: SyntaxWarning: "is" with a literal. Did you mea

ImportError: cannot import name 'models' from 'volatility3' (/usr/local/lib/python3.11/dist-packages/volatility3/__init__.py)

In [13]:
import math

import numpy as np


def get_estimator(price_data, window=30, trading_periods=252, clean=True):

    log_ho = (price_data['High'] / price_data['Open']).apply(np.log)
    log_lo = (price_data['Low'] / price_data['Open']).apply(np.log)
    log_co = (price_data['Close'] / price_data['Open']).apply(np.log)

    log_oc = (price_data['Open'] / price_data['Close'].shift(1)).apply(np.log)
    log_oc_sq = log_oc**2

    log_cc = (price_data['Close'] / price_data['Close'].shift(1)).apply(np.log)
    log_cc_sq = log_cc**2

    rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)

    close_vol = log_cc_sq.rolling(
        window=window,
        center=False
    ).sum() * (1.0 / (window - 1.0))
    open_vol = log_oc_sq.rolling(
        window=window,
        center=False
    ).sum() * (1.0 / (window - 1.0))
    window_rs = rs.rolling(
        window=window,
        center=False
    ).sum() * (1.0 / (window - 1.0))

    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    result = (open_vol + k * close_vol + (1 - k) * window_rs).apply(np.sqrt) * math.sqrt(trading_periods)

    if clean:
        return result.dropna()
    else:
        return result

In [14]:
import math

import numpy as np


def get_estimator(price_data, window=30, trading_periods=252, clean=True):

    log_hl = (price_data['High'] / price_data['Low']).apply(np.log)
    log_co = (price_data['Close'] / price_data['Open']).apply(np.log)

    rs = 0.5 * log_hl**2 - (2*math.log(2)-1) * log_co**2

    def f(v):
        return (trading_periods * v.mean())**0.5

    result = rs.rolling(window=window, center=False).apply(func=f)

    if clean:
        return result.dropna()
    else:
        return result

In [15]:
# Garman-Klass Volatility Model:




import math
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime

def get_estimator(price_data, window=30, trading_periods=252, clean=True):
    log_hl = (price_data['High'] / price_data['Low']).apply(np.log)
    log_co = (price_data['Close'] / price_data['Open']).apply(np.log)

    rs = 0.5 * log_hl**2 - (2*math.log(2)-1) * log_co**2

    def f(v):
        return (trading_periods * v.mean())**0.5

    result = rs.rolling(window=window, center=False).apply(func=f)

    if clean:
        return result.dropna()
    else:
        return result

# Function to fetch and analyze volatility for multiple stocks
def analyze_stocks(tickers, start_date, end_date, window=30, trading_periods=252):
    # Validate number of tickers
    if not 1 <= len(tickers) <= 10:
        raise ValueError("Please provide between 1 and 10 stock tickers.")

    # Dictionary to store results
    results = {}

    # Fetch data for each ticker
    for ticker in tickers:
        try:
            # Download stock data from Yahoo Finance
            stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)

            # Check if data is empty
            if stock_data.empty:
                print(f"No data retrieved for {ticker}. Skipping...")
                continue

            # Ensure required columns exist
            required_columns = ['High', 'Low', 'Close', 'Open']
            if not all(col in stock_data.columns for col in required_columns):
                print(f"Missing required columns for {ticker}. Skipping...")
                continue

            # Calculate volatility using the provided function
            volatility = get_estimator(stock_data, window=window, trading_periods=trading_periods, clean=True)

            # Store results in a DataFrame
            stock_data['Volatility'] = volatility

            # Add trading signals based on volatility thresholds
            stock_data['Signal'] = 'Hold'
            stock_data.loc[stock_data['Volatility'] < 0.2, 'Signal'] = 'Buy'
            stock_data.loc[stock_data['Volatility'] > 0.5, 'Signal'] = 'Sell'

            # Store the relevant columns
            results[ticker] = stock_data[['Close', 'Volatility', 'Signal']].copy()

        except Exception as e:
            print(f"Error processing {ticker}: {e}")

    return results

# Example usage
if __name__ == "__main__":
    # List of stock tickers (1–10)
    tickers = ['AAPL', 'MSFT', 'GOOGL', 'TGT', 'LOW', 'AMD']  # Add up to 10 tickers here

    # Editable date range
    start_date = '2024-07-23'  # 1 year prior to July 23, 2025
    end_date = '2025-07-23'    # Today: July 23, 2025

    # Analyze stocks
    try:
        results = analyze_stocks(tickers, start_date, end_date, window=30, trading_periods=252)

        # Print results for each stock
        for ticker, data in results.items():
            print(f"\nResults for {ticker}:")
            print(data.tail())  # Show last 5 rows for brevity
    except ValueError as ve:
        print(ve)
    except Exception as e:
        print(f"An error occurred: {e}")

/tmp/ipython-input-15-3849503953.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-15-3849503953.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-15-3849503953.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-15-3849503953.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-15-3849503953.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=Fa


Results for AAPL:
Price            Close Volatility Signal
Ticker            AAPL                  
Date                                    
2025-07-16  210.160004   0.206895   Hold
2025-07-17  210.020004   0.206707   Hold
2025-07-18  211.179993   0.203489   Hold
2025-07-21  212.479996   0.203635   Hold
2025-07-22  214.399994   0.201881   Hold

Results for MSFT:
Price            Close Volatility Signal
Ticker            MSFT                  
Date                                    
2025-07-16  505.619995   0.127823    Buy
2025-07-17  511.700012   0.129147    Buy
2025-07-18  510.049988   0.130896    Buy
2025-07-21  510.059998   0.131105    Buy
2025-07-22  505.269989   0.130229    Buy

Results for GOOGL:
Price            Close Volatility Signal
Ticker           GOOGL                  
Date                                    
2025-07-16  182.970001   0.237919   Hold
2025-07-17  183.580002   0.238076   Hold
2025-07-18  185.059998   0.239269   Hold
2025-07-21  190.100006   0.238331   Hold

Grok explains the above output, but note that initially only AAPL, MSFT, GOOGL were used in the model.TGT, LOW, AMD were added after first iteration of the code:

Structure of the Output

For each stock (AAPL, MSFT, GOOGL), the output is a table (pandas DataFrame) with the following columns:

Close: The closing price of the stock on the specified date.

Volatility: The annualized Garman-Klass volatility estimate, calculated using the get_estimator function over a 30-day rolling window (default window=30) and annualized based on 252 trading days (trading_periods=252).

Signal: A trading signal (Buy, Sell, or Hold) based on volatility thresholds:

Buy: Volatility < 0.2 (indicating low volatility, potentially stable or primed for a breakout).

Sell: Volatility > 0.5 (indicating high volatility, potentially risky).

Hold: Volatility between 0.2 and 0.5 (neutral, no strong action suggested).

The rows correspond to the last five trading days (July 16–22, 2025), with the Date as the index and the Ticker as a header for clarity.


Garman-Klass volatility eplained by Grok using AAPL as an example:

Volatility:

The Garman-Klass volatility ranges from 0.201881 to 0.206895 (approximately 20.2% to 20.7% annualized).

Volatility is calculated using the formula:

rs=0.5⋅(ln⁡(High/Low))2−(2⋅ln⁡(2)−1)⋅(ln⁡(Close/Open))2rs = 0.5 \cdot (\ln(\text{High}/\text{Low}))^2 - (2 \cdot \ln(2) - 1) \cdot (\ln(\text{Close}/\text{Open}))^2rs = 0.5 \cdot (\ln(\text{High}/\text{Low}))^2 - (2 \cdot \ln(2) - 1) \cdot (\ln(\text{Close}/\text{Open}))^2

followed by annualizing over a 30-day rolling window: 252⋅mean(rs)\sqrt{252 \cdot \text{mean}(rs)}\sqrt{252 \cdot \text{mean}(rs)}
.
The values are slightly above the 0.2 threshold, indicating moderate price fluctuations. The slight decrease in volatility from 0.206895 (July 16) to 0.201881 (July 22) suggests stabilizing price movements.






In [18]:
# Yang-Zhang Volatility Model:

import math
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime

def get_estimator(price_data, window=30, trading_periods=252, clean=True):
    log_ho = (price_data['High'] / price_data['Open']).apply(np.log)
    log_lo = (price_data['Low'] / price_data['Open']).apply(np.log)
    log_co = (price_data['Close'] / price_data['Open']).apply(np.log)

    log_oc = (price_data['Open'] / price_data['Close'].shift(1)).apply(np.log)
    log_oc_sq = log_oc**2

    log_cc = (price_data['Close'] / price_data['Close'].shift(1)).apply(np.log)
    log_cc_sq = log_cc**2

    rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)

    close_vol = log_cc_sq.rolling(
        window=window,
        center=False
    ).sum() * (1.0 / (window - 1.0))
    open_vol = log_oc_sq.rolling(
        window=window,
        center=False
    ).sum() * (1.0 / (window - 1.0))
    window_rs = rs.rolling(
        window=window,
        center=False
    ).sum() * (1.0 / (window - 1.0))

    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    result = (open_vol + k * close_vol + (1 - k) * window_rs).apply(np.sqrt) * math.sqrt(trading_periods)

    if clean:
        return result.dropna()
    else:
        return result

# Function to fetch and analyze volatility for multiple stocks
def analyze_stocks(tickers, start_date, end_date, window=30, trading_periods=252):
    # Validate number of tickers
    if not 1 <= len(tickers) <= 10:
        raise ValueError("Please provide between 1 and 10 stock tickers.")

    # Dictionary to store results
    results = {}

    # Fetch data for each ticker
    for ticker in tickers:
        try:
            # Download stock data from Yahoo Finance
            stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)

            # Check if data is empty
            if stock_data.empty:
                print(f"No data retrieved for {ticker}. Skipping...")
                continue

            # Ensure required columns exist
            required_columns = ['High', 'Low', 'Close', 'Open']
            if not all(col in stock_data.columns for col in required_columns):
                print(f"Missing required columns for {ticker}. Skipping...")
                continue

            # Calculate volatility using the provided function
            volatility = get_estimator(stock_data, window=window, trading_periods=trading_periods, clean=True)

            # Store results in a DataFrame
            stock_data['Volatility'] = volatility

            # Add trading signals based on volatility thresholds
            stock_data['Signal'] = 'Hold'
            stock_data.loc[stock_data['Volatility'] < 0.2, 'Signal'] = 'Buy'
            stock_data.loc[stock_data['Volatility'] > 0.5, 'Signal'] = 'Sell'

            # Store the relevant columns
            results[ticker] = stock_data[['Close', 'Volatility', 'Signal']].copy()

        except Exception as e:
            print(f"Error processing {ticker}: {e}")

    return results

# Example usage
if __name__ == "__main__":
    # List of stock tickers (1–10)
    tickers = ['V', 'MSFT', 'MA', 'TGT', 'LOW', 'AMD', 'FCX', 'GLD', 'NLY', 'JPM' ]  # Add up to 10 tickers here

    # Editable date range
    start_date = '2024-07-23'  # 1 year prior to July 23, 2025
    end_date = '2025-07-23'    # Today: July 23, 2025

    # Analyze stocks
    try:
        results = analyze_stocks(tickers, start_date, end_date, window=30, trading_periods=252)

        # Print results for each stock
        for ticker, data in results.items():
            print(f"\nResults for {ticker}:")
            print(data.tail())  # Show last 5 rows for brevity
    except ValueError as ve:
        print(ve)
    except Exception as e:
        print(f"An error occurred: {e}")

/tmp/ipython-input-18-992792733.py:56: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-18-992792733.py:56: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-18-992792733.py:56: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-18-992792733.py:56: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-18-992792733.py:56: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)



Results for V:
Price            Close Volatility Signal
Ticker               V                  
Date                                    
2025-07-16  349.899994   0.244660   Hold
2025-07-17  349.809998   0.243279   Hold
2025-07-18  349.049988   0.243052   Hold
2025-07-21  350.940002   0.243462   Hold
2025-07-22  351.859985   0.243262   Hold

Results for MSFT:
Price            Close Volatility Signal
Ticker            MSFT                  
Date                                    
2025-07-16  505.619995   0.143071    Buy
2025-07-17  511.700012   0.144404    Buy
2025-07-18  510.049988   0.146591    Buy
2025-07-21  510.059998   0.147537    Buy
2025-07-22  505.269989   0.145875    Buy

Results for MA:
Price            Close Volatility Signal
Ticker              MA                  
Date                                    
2025-07-16  555.520020   0.243800   Hold
2025-07-17  555.609985   0.242500   Hold
2025-07-18  552.659973   0.242387   Hold
2025-07-21  554.650024   0.242681   Hold
2025-

/tmp/ipython-input-18-992792733.py:56: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-18-992792733.py:56: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-18-992792733.py:56: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)


Grok explains difference between the two models:

Conclusion

**Differences**: Yang-Zhang is more comprehensive, incorporating overnight and close-to-close volatility, while Garman-Klass focuses solely on intraday volatility, making it simpler but less complete.

**Recommendation**: Yang-Zhang is generally more accurate, reliable, and suitable for trading due to its inclusion of overnight volatility, which is critical for most stocks and trading strategies. Use Garman-Klass for intraday-focused strategies or when computational simplicity is prioritized, but Yang-Zhang is preferred for broader applications, especially in volatile or gap-prone markets.



In [21]:
import datetime
import os
import pandas
import numpy
from scipy.stats import norm
import statsmodels.api as sm
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import yfinance as yf

from volatility import models  # Assumed to contain estimator functions

ESTIMATORS = [
    'GarmanKlass',
    'HodgesTompkins',
    'Kurtosis',
    'Parkinson',
    'Raw',
    'RogersSatchell',
    'Skew',
    'YangZhang'
]
PRICE_COLUMNS = {'Open', 'High', 'Low', 'Close'}

def array_to_dataframe(ndarray):
    return pandas.DataFrame(
        ndarray,
        columns=['Open', 'High', 'Low', 'Close']
    )

class VolatilityEstimator(object):
    def __init__(self, price_data, estimator, bench_data=None):
        if not isinstance(price_data, numpy.ndarray) and not isinstance(price_data, pandas.DataFrame):
            raise ValueError('price_data must be of type numpy.ndarray or pandas.DataFrame')
        if isinstance(price_data, numpy.ndarray) and price_data.shape[1] != 4:  # Corrected shape check
            raise ValueError('price_data of type numpy.ndarray shape of (r, 4)')
        if isinstance(price_data, pandas.DataFrame) and not PRICE_COLUMNS.issubset(price_data.columns):
            raise ValueError('price_data requires Open, High, Low, Close')
        if not hasattr(price_data, 'symbol') or price_data.symbol is None or price_data.symbol == '':
            raise ValueError('Symbol required as property of price_data')
        if estimator not in ESTIMATORS:
            raise ValueError('Acceptable volatility model is required')

        if isinstance(price_data, numpy.ndarray):
            price_data = array_to_dataframe(price_data)
            price_data.symbol = '-NA-'
            start = price_data.index[0]
            end = price_data.index[0]
        else:
            start = price_data.index[0].to_pydatetime().strftime('%Y-%m-%d')
            end = price_data.index[-1].to_pydatetime().strftime('%Y-%m-%d')

        if bench_data is not None:
            if price_data.shape != bench_data.shape:
                raise ValueError('price_data and bench_data must be same shape')
            if not isinstance(bench_data, numpy.ndarray) and not isinstance(bench_data, pandas.DataFrame):
                raise ValueError('bench_data must be of type numpy.ndarray or pandas.DataFrame')
            if isinstance(bench_data, numpy.ndarray) and bench_data.shape[1] != 4:  # Corrected shape check
                raise ValueError('bench_data of type numpy.ndarray shape of (r, 4)')
            if isinstance(bench_data, pandas.DataFrame) and not PRICE_COLUMNS.issubset(bench_data.columns):
                raise ValueError('bench_data requires Open, High, Low, Close')
            if not hasattr(bench_data, 'symbol') or bench_data.symbol is None or bench_data.symbol == '':
                raise ValueError('Symbol required as property of bench_data')

            if isinstance(bench_data, numpy.ndarray):
                bench_data = array_to_dataframe(bench_data)
                bench_data.symbol = '-NA-'

            self._bench_data = bench_data
            self._bench_symbol = bench_data.symbol

        self._price_data = price_data
        self._symbol = price_data.symbol
        self._start = start
        self._end = end
        self._estimator = estimator

        matplotlib.rc('image', origin='upper')
        matplotlib.rcParams['font.size'] = '11'
        matplotlib.rcParams['grid.color'] = 'lightgrey'
        matplotlib.rcParams['grid.linestyle'] = '-'
        matplotlib.rcParams['figure.subplot.left'] = 0.1
        matplotlib.rcParams['figure.subplot.bottom'] = 0.13
        matplotlib.rcParams['figure.subplot.right'] = 0.9
        matplotlib.rcParams['figure.subplot.top'] = 0.9

    def _get_estimator(self, window, price_data, clean=True):
        return getattr(models, self._estimator).get_estimator(
            price_data=price_data,
            window=window,
            clean=clean
        )

    def cones(self, windows=[30, 60, 90, 120], quantiles=[0.25, 0.75]):
        price_data = self._price_data
        if len(windows) < 2:
            raise ValueError('Two or more window periods required')
        if len(quantiles) != 2:
            raise ValueError('A two element list of quantiles is required, lower and upper')
        if quantiles[0] + quantiles[1] != 1.0:
            raise ValueError('The sum of the quantiles must equal 1.0')
        if quantiles[0] > quantiles[1]:
            raise ValueError('The lower quantiles (first element) must be less than the upper quantile (second element)')

        max_ = []
        min_ = []
        top_q = []
        median = []
        bottom_q = []
        realized = []
        data = []

        for window in windows:
            estimator = self._get_estimator(window=window, price_data=price_data)
            max_.append(estimator.max())
            top_q.append(estimator.quantile(quantiles[1]))
            median.append(estimator.median())
            bottom_q.append(estimator.quantile(quantiles[0]))
            min_.append(estimator.min())
            realized.append(estimator[-1])
            data.append(estimator)

        if self._estimator in ["Skew", "Kurtosis"]:
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        bottom, height = 0.2, 0.7
        left_h = left + width + 0.02
        rect_cones = [left, bottom, width, height]
        rect_box = [left_h, bottom, 0.17, height]
        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        cones.plot(windows, max_, label="Max")
        cones.plot(windows, top_q, label=str(int(quantiles[1]*100)) + " Prctl")
        cones.plot(windows, median, label="Median")
        cones.plot(windows, bottom_q, label=str(int(quantiles[0]*100)) + " Prctl")
        cones.plot(windows, min_, label="Min")
        cones.plot(windows, realized, 'r-.', label="Realized")
        cones.set_xticks(windows)
        cones.set_xlim((windows[0]-5, windows[-1]+5))
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))
        cones.grid(True, axis='y', which='major', alpha=0.5)
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
        box.boxplot(data, notch=1, sym='+')
        box.plot([i for i in range(1, len(windows)+1)], realized, color='r', marker='*', markeredgecolor='k')
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))
        box.yaxis.tick_right()
        box.grid(True, axis='y', which='major', alpha=0.5)
        return fig, plt

    def rolling_quantiles(self, window=30, quantiles=[0.25, 0.75]):
        price

ModuleNotFoundError: No module named 'volatility'

In [22]:
import datetime
import os
import pandas
import numpy
from scipy.stats import norm
import statsmodels.api as sm
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import yfinance as yf

# Define estimator functions directly in the script
class GarmanKlass:
    @staticmethod
    def get_estimator(price_data, window=30, trading_periods=252, clean=True):
        log_hl = (price_data['High'] / price_data['Low']).apply(np.log)
        log_co = (price_data['Close'] / price_data['Open']).apply(np.log)
        rs = 0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2
        result = rs.rolling(window=window, center=False).mean().apply(np.sqrt) * np.sqrt(trading_periods)
        return result.dropna() if clean else result

class YangZhang:
    @staticmethod
    def get_estimator(price_data, window=30, trading_periods=252, clean=True):
        log_ho = (price_data['High'] / price_data['Open']).apply(np.log)
        log_lo = (price_data['Low'] / price_data['Open']).apply(np.log)
        log_co = (price_data['Close'] / price_data['Open']).apply(np.log)
        log_oc = (price_data['Open'] / price_data['Close'].shift(1)).apply(np.log)
        log_oc_sq = log_oc**2
        log_cc = (price_data['Close'] / price_data['Close'].shift(1)).apply(np.log)
        log_cc_sq = log_cc**2
        rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)
        close_vol = log_cc_sq.rolling(window=window, center=False).sum() * (1.0 / (window - 1.0))
        open_vol = log_oc_sq.rolling(window=window, center=False).sum() * (1.0 / (window - 1.0))
        window_rs = rs.rolling(window=window, center=False).sum() * (1.0 / (window - 1.0))
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        result = (open_vol + k * close_vol + (1 - k) * window_rs).apply(np.sqrt) * np.sqrt(trading_periods)
        return result.dropna() if clean else result

ESTIMATORS = ['GarmanKlass', 'YangZhang']  # Limited to implemented estimators
PRICE_COLUMNS = {'Open', 'High', 'Low', 'Close'}

def array_to_dataframe(ndarray):
    return pandas.DataFrame(
        ndarray,
        columns=['Open', 'High', 'Low', 'Close']
    )

class VolatilityEstimator(object):
    def __init__(self, price_data, estimator, bench_data=None):
        if not isinstance(price_data, numpy.ndarray) and not isinstance(price_data, pandas.DataFrame):
            raise ValueError('price_data must be of type numpy.ndarray or pandas.DataFrame')
        if isinstance(price_data, numpy.ndarray) and price_data.shape[1] != 4:
            raise ValueError('price_data of type numpy.ndarray shape of (r, 4)')
        if isinstance(price_data, pandas.DataFrame) and not PRICE_COLUMNS.issubset(price_data.columns):
            raise ValueError('price_data requires Open, High, Low, Close')
        if not hasattr(price_data, 'symbol') or price_data.symbol is None or price_data.symbol == '':
            raise ValueError('Symbol required as property of price_data')
        if estimator not in ESTIMATORS:
            raise ValueError(f'Acceptable volatility model is required. Choose from {ESTIMATORS}')

        if isinstance(price_data, numpy.ndarray):
            price_data = array_to_dataframe(price_data)
            price_data.symbol = '-NA-'
            start = price_data.index[0]
            end = price_data.index[0]
        else:
            start = price_data.index[0].to_pydatetime().strftime('%Y-%m-%d')
            end = price_data.index[-1].to_pydatetime().strftime('%Y-%m-%d')

        if bench_data is not None:
            if price_data.shape != bench_data.shape:
                raise ValueError('price_data and bench_data must be same shape')
            if not isinstance(bench_data, numpy.ndarray) and not isinstance(bench_data, pandas.DataFrame):
                raise ValueError('bench_data must be of type numpy.ndarray or pandas.DataFrame')
            if isinstance(bench_data, numpy.ndarray) and bench_data.shape[1] != 4:
                raise ValueError('bench_data of type numpy.ndarray shape of (r, 4)')
            if isinstance(bench_data, pandas.DataFrame) and not PRICE_COLUMNS.issubset(bench_data.columns):
                raise ValueError('bench_data requires Open, High, Low, Close')
            if not hasattr(bench_data, 'symbol') or bench_data.symbol is None or bench_data.symbol == '':
                raise ValueError('Symbol required as property of bench_data')

            if isinstance(bench_data, numpy.ndarray):
                bench_data = array_to_dataframe(bench_data)
                bench_data.symbol = '-NA-'

            self._bench_data = bench_data
            self._bench_symbol = bench_data.symbol

        self._price_data = price_data
        self._symbol = price_data.symbol
        self._start = start
        self._end = end
        self._estimator = estimator

        matplotlib.rc('image', origin='upper')
        matplotlib.rcParams['font.size'] = '11'
        matplotlib.rcParams['grid.color'] = 'lightgrey'
        matplotlib.rcParams['grid.linestyle'] = '-'
        matplotlib.rcParams['figure.subplot.left'] = 0.1
        matplotlib.rcParams['figure.subplot.bottom'] = 0.13
        matplotlib.rcParams['figure.subplot.right'] = 0.9
        matplotlib.rcParams['figure.subplot.top'] = 0.9

    def _get_estimator(self, window, price_data, clean=True):
        return globals()[self._estimator].get_estimator(
            price_data=price_data,
            window=window,
            clean=clean
        )

    def cones(self, windows=[30, 60, 90, 120], quantiles=[0.25, 0.75]):
        price_data = self._price_data
        if len(windows) < 2:
            raise ValueError('Two or more window periods required')
        if len(quantiles) != 2:
            raise ValueError('A two element list of quantiles is required, lower and upper')
        if quantiles[0] + quantiles[1] != 1.0:
            raise ValueError('The sum of the quantiles must equal 1.0')
        if quantiles[0] > quantiles[1]:
            raise ValueError('The lower quantiles (first element) must be less than the upper quantile (second element)')

        max_ = []
        min_ = []
        top_q = []
        median = []
        bottom_q = []
        realized = []
        data = []

        for window in windows:
            estimator = self._get_estimator(window=window, price_data=price_data)
            max_.append(estimator.max())
            top_q.append(estimator.quantile(quantiles[1]))
            median.append(estimator.median())
            bottom_q.append(estimator.quantile(quantiles[0]))
            min_.append(estimator.min())
            realized.append(estimator[-1])
            data.append(estimator)

        if self._estimator in ["Skew", "Kurtosis"]:
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        bottom, height = 0.2, 0.7
        left_h = left + width + 0.02
        rect_cones = [left, bottom, width, height]
        rect_box = [left_h, bottom, 0.17, height]
        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        cones.plot(windows, max_, label="Max")
        cones.plot(windows, top_q, label=str(int(quantiles[1]*100)) + " Prctl")
        cones.plot(windows, median, label="Median")
        cones.plot(windows, bottom_q, label=str(int(quantiles[0]*100)) + " Prctl")
        cones.plot(windows, min_, label="Min")
        cones.plot(windows, realized, 'r-.', label="Realized")
        cones.set_xticks(windows)
        cones.set_xlim((windows[0]-5, windows[-1]+5))
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))
        cones.grid(True, axis='y', which='major', alpha=0.5)
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
        box.boxplot(data, notch=1, sym='+')
        box.plot([i for i in range(1, len(windows)+1)], realized, color='r', marker='*', markeredgecolor='k')
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))
        box.yaxis.tick_right()
        box.grid(True, axis='y', which='major', alpha=0.5)
        return fig, plt

    def rolling_quantiles(self, window=30, quantiles=[0.25, 0.75]):
        price_data = self._price_data
        if len(quantiles) != 2:
            raise ValueError('A two element list of quantiles is required, lower and upper')
        if quantiles[0] + quantiles[1] != 1.0:
            raise ValueError('The sum of the quantiles must equal 1.0')
        if quantiles[0] > quantiles[1]:
            raise ValueError('The lower quantiles (first element) must be less than the upper quantile (second element)')

        estimator = self._get_estimator(window=window, price_data=price_data)
        date = estimator.index
        top_q = estimator.rolling(window=window, center=False).quantile(quantiles[1])
        median = estimator.rolling(window=window, center=False).median()
        bottom_q = estimator.rolling(window=window, center=False).quantile(quantiles[0])
        realized = estimator
        last = estimator[-1]

        if self._estimator in ["Skew", "Kurtosis"]:
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        bottom, height = 0.2, 0.7
        left_h = left + width + 0.02
        rect_cones = [left, bottom, width, height]
        rect_box = [left_h, bottom, 0.17, height]
        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        cones.plot(date, top_q, label=str(int(quantiles[1]*100)) + " Prctl")
        cones.plot(date, median, label="Median")
        cones.plot(date, bottom_q, label=str(int(quantiles[0]*100)) + " Prctl")
        cones.plot(date, realized, 'r-.', label="Realized")
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))
        cones.grid(True, axis='y', which='major', alpha=0.5)
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
        box.boxplot(realized, notch=1, sym='+')
        box.plot(1, last, color='r', marker='*', markeredgecolor='k')
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))
        box.yaxis.tick_right()
        box.grid(True, axis='y', which='major', alpha=0.5)
        return fig, plt

    def rolling_extremes(self, window=30):
        price_data = self._price_data
        estimator = self._get_estimator(window=window, price_data=price_data)
        date = estimator.index
        max_ = estimator.rolling(window=window, center=False).max()
        min_ = estimator.rolling(window=window, center=False).min()
        realized = estimator
        last = estimator[-1]

        if self._estimator in ["Skew", "Kurtosis"]:
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        bottom, height = 0.2, 0.7
        left_h = left + width + 0.02
        rect_cones = [left, bottom, width, height]
        rect_box = [left_h, bottom, 0.17, height]
        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        cones.plot(date, max_, label="Max")
        cones.plot(date, min_, label="Min")
        cones.plot(date, realized, 'r-.', label="Realized")
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))
        cones.grid(True, axis='y', which='major', alpha=0.5)
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
        box.boxplot(realized, notch=1, sym='+')
        box.plot(1, last, color='r', marker='*', markeredgecolor='k')
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))
        box.yaxis.tick_right()
        box.grid(True, axis='y', which='major', alpha=0.5)
        return fig, plt

    def rolling_descriptives(self, window=30):
        price_data = self._price_data
        estimator = self._get_estimator(window=window, price_data=price_data)
        date = estimator.index
        mean = estimator.rolling(window=window, center=False).mean()
        std = estimator.rolling(window=window, center=False).std()
        z_score = (estimator - mean) / std
        realized = estimator
        last = estimator[-1]

        if self._estimator in ["Skew", "Kurtosis"]:
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, 0.65
        left_h = left + width + 0.02
        rect_cones = [left, 0.35, width, 0.55]
        rect_box = [left_h, 0.15, 0.17, 0.75]
        rect_z = [left, 0.15, width, 0.15]
        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)
        z = plt.axes(rect_z)

        cones.plot(date, mean, label="Mean")
        cones.plot(date, std, label="Std. Dev.")
        cones.plot(date, realized, 'r-.', label="Realized")
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))
        cones.grid(True, axis='y', which='major', alpha=0.5)
        cones.set_title(self._estimator + ' (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')
        pos = cones.get_position()
        cones.set_position([pos.x0, pos.y0 + pos.height * 0.1, pos.width, pos.height * 0.9])
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
        box.boxplot(realized, notch=1, sym='+')
        box.plot(1, last, color='r', marker='*', markeredgecolor='k')
        locs = box.get_yticks()
        box.set_yticklabels(map(f, locs))
        box.yaxis.tick_right()
        box.grid(True, axis='y', which='major', alpha=0.5)
        z.plot(date, z_score, 'm-', label="Z-Score")
        z.grid(True, axis='y', which='major', alpha=0.5)
        z.axhline(0, 0, 1, linestyle='-', linewidth=1.0, color='black')
        z.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)
        return fig, plt

    def histogram(self, window=90, bins=100, normed=True):
        price_data = self._price_data
        estimator = self._get_estimator(window=window, price_data=price_data)
        mean = estimator.mean()
        std = estimator.std()
        last = estimator[-1]

        fig = plt.figure(figsize=(8, 6))
        n, bins, patches = plt.hist(estimator, bins, density=normed, facecolor='blue', alpha=0.25)
        if normed:
            y = norm.pdf(bins, mean, std)
            plt.plot(bins, y, 'g--', linewidth=1)
        plt.axvline(last, 0, 1, linestyle='-', linewidth=1.5, color='r')
        plt.grid(True, axis='y', which='major', alpha=0.5)
        plt.title('Distribution of ' + self._estimator + ' estimator values (' + self._symbol + ', daily ' + self._start + ' to ' + self._end + ')')
        return fig, plt

    def benchmark_compare(self, window=90):
        price_data = self._price_data
        bench_data = self._bench_data
        y = self._get_estimator(window=window, price_data=price_data)
        x = self._get_estimator(window=window, price_data=bench_data)
        date = y.index
        ratio = y / x

        if self._estimator in ["Skew", "Kurtosis"]:
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        fig = plt.figure(figsize=(8, 6))
        fig.autofmt_xdate()
        left, width = 0.07, .9
        rect_cones = [left, 0.4, width, .5]
        rect_box = [left, 0.15, width, 0.15]
        cones = plt.axes(rect_cones)
        box = plt.axes(rect_box)

        cones.plot(date, y, label=self._symbol.upper())
        cones.plot(date, x, label=self._bench_symbol)
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))
        cones.grid(True, axis='y', which='major', alpha=0.5)
        cones.set_title(self._estimator + ' (' + self._symbol + ' v. ' + self._bench_symbol + ', daily ' + self._start + ' to ' + self._end + ')')
        cones.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
        box.plot(date, ratio, label=self._symbol.upper() + '/' + self._bench_symbol)
        box.set_ylim((ratio.min() - 0.05, ratio.max() + 0.05))
        box.fill_between(date, ratio, 0, color='blue', alpha=0.25)
        box.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)
        return fig, plt

    def benchmark_correlation(self, window=90):
        price_data = self._price_data
        bench_data = self._bench_data
        y = self._get_estimator(window=window, price_data=price_data)
        x = self._get_estimator(window=window, price_data=bench_data)
        date = y.index
        corr = x.rolling(window=window).corr(other=y)

        if self._estimator in ["Skew", "Kurtosis"]:
            f = lambda x: "%i" % round(x, 0)
        else:
            f = lambda x: "%i%%" % round(x*100, 0)

        fig = plt.figure(figsize=(8, 6))
        cones = plt.axes()
        cones.plot(date, corr)
        cones.set_ylim((corr.min() - 0.05, corr.max() + 0.05))
        locs = cones.get_yticks()
        cones.set_yticklabels(map(f, locs))
        cones.grid(True, axis='y', which='major', alpha=0.5)
        cones.set_title(self._estimator + ' (Correlation of ' + self._symbol + ' v. ' + self._bench_symbol + ', daily ' + self._start + ' to ' + self._end + ')')
        return fig, plt

    def benchmark_regression(self, window=90):
        price_data = self._price_data
        bench_data = self._bench_data
        y = self._get_estimator(window=window, price_data=price_data)
        X = self._get_estimator(window=window, price_data=bench_data)
        model = sm.OLS(y, X)
        results = model.fit()
        return results.summary()

    def term_sheet(self, window=30, windows=[30, 60, 90, 120], quantiles=[0.25, 0.75], bins=100, normed=True, open=False):
        cones_fig, cones_plt = self.cones(windows=windows, quantiles=quantiles)
        rolling_quantiles_fig, rolling_quantiles_plt = self.rolling_quantiles(window=window, quantiles=quantiles)
        rolling_extremes_fig, rolling_extremes_plt = self.rolling_extremes(window=window)
        rolling_descriptives_fig, rolling_descriptives_plt = self.rolling_descriptives(window=window)
        histogram_fig, histogram_plt = self.histogram(window=window, bins=bins, normed=normed)
        benchmark_compare_fig, benchmark_compare_plt = self.benchmark_compare(window=window)
        benchmark_corr_fig, benchmark_corr_plt = self.benchmark_correlation(window=window)
        benchmark_regression = self.benchmark_regression(window=window)

        filename = self._symbol.upper() + '_termsheet_' + datetime.datetime.today().strftime("%Y%m%d") + '.pdf'
        fn = os.path.abspath(os.path.join('term-sheets', filename))
        os.makedirs('term-sheets', exist_ok=True)
        pp = PdfPages(fn)

        pp.savefig(cones_fig)
        pp.savefig(rolling_quantiles_fig)
        pp.savefig(rolling_extremes_fig)
        pp.savefig(rolling_descriptives_fig)
        pp.savefig(histogram_fig)
        pp.savefig(benchmark_compare_fig)
        pp.savefig(benchmark_corr_fig)

        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111)
        ax.text(0, .2, benchmark_regression, family='monospace', fontsize=9)
        plt.axis('off')
        fig.tight_layout()
        pp.savefig(fig)
        pp.close()

        print('%s output complete' % filename)

def analyze_stocks(tickers, start_date, end_date, estimator='YangZhang', window=30, trading_periods=252, bench_ticker=None):
    if not 1 <= len(tickers) <= 10:
        raise ValueError("Please provide between 1 and 10 stock tickers.")

    results = {}

    # Fetch benchmark data if provided
    bench_data = None
    if bench_ticker:
        try:
            bench_data = yf.download(bench_ticker, start=start_date, end=end_date, progress=False)
            if bench_data.empty:
                print(f"No data retrieved for benchmark {bench_ticker}. Proceeding without benchmark...")
                bench_data = None
            else:
                bench_data.symbol = bench_ticker.upper()
        except Exception as e:
            print(f"Error fetching benchmark {bench_ticker}: {e}. Proceeding without benchmark...")
            bench_data = None

    # Fetch and process data for each ticker
    for ticker in tickers:
        try:
            # Download stock data from Yahoo Finance
            stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)

            if stock_data.empty:
                print(f"No data retrieved for {ticker}. Skipping...")
                continue

            if not all(col in stock_data.columns for col in PRICE_COLUMNS):
                print(f"Missing required columns for {ticker}. Skipping...")
                continue

            # Set symbol attribute
            stock_data.symbol = ticker.upper()

            # Initialize VolatilityEstimator
            vol_estimator = VolatilityEstimator(
                price_data=stock_data,
                estimator=estimator,
                bench_data=bench_data
            )

            # Calculate volatility
            volatility = vol_estimator._get_estimator(window=window, price_data=stock_data, clean=True)

            # Store results
            stock_data['Volatility'] = volatility
            stock_data['Signal'] = 'Hold'
            stock_data.loc[stock_data['Volatility'] < 0.2, 'Signal'] = 'Buy'
            stock_data.loc[stock_data['Volatility'] > 0.5, 'Signal'] = 'Sell'

            results[ticker] = stock_data[['Close', 'Volatility', 'Signal']].copy()

            # Generate term sheet
            try:
                vol_estimator.term_sheet(window=window, windows=[30, 60, 90, 120], quantiles=[0.25, 0.75], bins=100, normed=True)
            except Exception as e:
                print(f"Error generating term sheet for {ticker}: {e}")

        except Exception as e:
            print(f"Error processing {ticker}: {e}")

    return results

if __name__ == "__main__":
    tickers = ['AAPL', 'MSFT', 'GOOGL']  # Add up to 10 tickers
    start_date = '2024-07-23'  # Editable: 1 year prior to July 23, 2025
    end_date = '2025-07-23'    # Editable: Today
    estimator = 'YangZhang'     # Choose from ['GarmanKlass', 'YangZhang']
    bench_ticker = 'SPY'        # Optional benchmark (e.g., S&P 500 ETF)

    try:
        results = analyze_stocks(tickers, start_date, end_date, estimator=estimator, window=30, trading_periods=252, bench_ticker=bench_ticker)
        for ticker, data in results.items():
            print(f"\nResults for {ticker}:")
            print(data.tail())  # Show last 5 rows
    except ValueError as ve:
        print(ve)
    except Exception as e:
        print(f"An error occurred: {e}")

/tmp/ipython-input-22-2026417217.py:440: FutureWarning: YF.download() has changed argument auto_adjust default to True
  bench_data = yf.download(bench_ticker, start=start_date, end=end_date, progress=False)


Error processing AAPL: price_data requires Open, High, Low, Close
Error processing MSFT: price_data requires Open, High, Low, Close
Error processing GOOGL: price_data requires Open, High, Low, Close


/tmp/ipython-input-22-2026417217.py:454: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-22-2026417217.py:454: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-22-2026417217.py:454: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
